<img src="https://res.cloudinary.com/dtizipxds/image/upload/q_auto/f_auto/v1776264397/banner_yvwajv.png" width="100%">


In [1]:
%pip install numpy pandas matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.


# Preprocessing (Complete but Simple)

This notebook is a full preprocessing walkthrough designed for beginners.
You will start from raw data, fix common quality issues, engineer useful features, and finish with a model-ready train/test split.

What you will practice:
- inspecting a new dataset quickly
- handling missing values with several strategies
- removing duplicate records safely
- converting and using date columns
- encoding categorical variables for machine learning
- capping outliers with a robust rule
- scaling numeric features
- preparing `X` and `y` for model training

Tip: after each section, compare the output with the previous section so you can see exactly what changed.


In [2]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split


## 1) Create a Realistic Practice Dataset

We load the `wine` dataset, then reshape it into a customer-style table to simulate a real business case.
After that, we intentionally inject data problems (missing values, duplicates, inconsistent records).

Why this matters:
- real-world datasets are rarely clean
- preprocessing is easier to learn when issues are realistic
- we can test multiple cleaning strategies on the same data

Goal of this step: build a controlled "messy" dataset that we will clean in the next sections.


In [3]:
wine = load_wine(as_frame=True)
base = wine.frame.copy()

df = pd.DataFrame({
    "customer_id": np.arange(1, len(base) + 1),
    "age": (base["color_intensity"] * 3 + 18).round(),
    "income": (base["proline"] * 120 + 25000).round(),
    "spent": (base["total_phenols"] * 900 + 300).round(),
    "purchases": (base["flavanoids"] * 2 + 3).round(),
    "city": np.where(base["proline"] > base["proline"].median(), "Algiers", "Oran"),
    "education": np.where(base["alcohol"] > 13.5, "Master", "Bachelor"),
    "joined_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(base.index * 2, unit="D"),
    "last_login": pd.to_datetime("2025-02-01") + pd.to_timedelta(base.index % 20, unit="D"),
    "churned": np.where(base["target"] == 2, "Yes", "No")
})

# Add real-world issues on purpose

df.loc[[2, 12, 40], "income"] = np.nan
df.loc[[8, 33], "age"] = np.nan
df.loc[[5], "city"] = None
df.loc[[15], "education"] = None
df.loc[[22], "spent"] = np.nan
df.loc[[9], "last_login"] = None

# Exact duplicate

df = pd.concat([df, df.iloc[[0]]], ignore_index=True)

# Duplicate customer_id with newer login
dup = df.iloc[[25]].copy()
dup["last_login"] = pd.to_datetime("2025-03-05")
df = pd.concat([df, dup], ignore_index=True)

df.head()


,customer_id,age,income,spent,purchases,city,education,joined_date,last_login,churned
0,1,35.0,152800.0,2820.0,9.0,Algiers,Master,2024-01-01,2025-02-01,No
1,2,31.0,151000.0,2685.0,9.0,Algiers,Bachelor,2024-01-03,2025-02-02,No
2,3,35.0,NaN,2820.0,9.0,Algiers,Bachelor,2024-01-05,2025-02-03,No
3,4,41.0,202600.0,3765.0,10.0,Algiers,Master,2024-01-07,2025-02-04,No
4,5,31.0,113200.0,2820.0,8.0,Algiers,Bachelor,2024-01-09,2025-02-05,No


## 2) First Inspection

Before applying any fix, always profile the dataset first.
Here we check shape, column types, missing values, and duplicates.

Why this matters:
- helps choose the right cleaning method
- prevents blind transformations
- gives a baseline to verify improvement later

Output to focus on:
- which columns contain nulls
- whether date columns are parsed correctly
- how many duplicates exist


In [4]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing values:\n", df.isna().sum())
print("\nExact duplicates:", df.duplicated().sum())


Shape: (180, 10)

Dtypes:
 customer_id             int32
age                   float64
income                float64
spent                 float64
purchases             float64
city                   object
education              object
joined_date    datetime64[ns]
last_login     datetime64[ns]
churned                object
dtype: object

Missing values:
 customer_id    0
age            2
income         3
spent          1
purchases      0
city           1
education      1
joined_date    0
last_login     1
churned        0
dtype: int64

Exact duplicates: 1


## 3) Missing Values: Method 1 (Drop Rows)

This method removes rows with missing values in a chosen column.
It is simple and sometimes effective, but it can reduce dataset size.

Use it when:
- only a small fraction of rows is missing
- dropped rows are not critical
- dataset is large enough to absorb row loss

Risk: if many rows are dropped, model quality may degrade.


In [5]:
m1_drop = df.dropna(subset=["income"]).copy()
print("Rows before:", len(df), "| after drop income nulls:", len(m1_drop))


Rows before: 180 | after drop income nulls: 177


## Missing Values: Method 2 (Fill with a Constant)

Here we replace missing categorical values with a fixed label like `Unknown`.

Use it when:
- a safe default category exists
- you want to preserve all rows
- missingness itself may carry information

This is common for text/categorical columns such as city, segment, or device type.


In [6]:
m2_constant = df.copy()
m2_constant["city"] = m2_constant["city"].fillna("Unknown")
print(m2_constant[["city"]].isna().sum())


city    0
dtype: int64


## Missing Values: Method 3 (Median/Mode)

This is one of the most practical default strategies:
- numeric features -> median (less sensitive to outliers than mean)
- categorical features -> mode (most common category)

Why it is popular:
- easy to explain
- stable in many datasets
- keeps dataset size unchanged

In many projects, this method is a strong baseline.


In [7]:
m3_stat = df.copy()

m3_stat["income"] = m3_stat["income"].fillna(m3_stat["income"].median())
m3_stat["age"] = m3_stat["age"].fillna(m3_stat["age"].median())
m3_stat["spent"] = m3_stat["spent"].fillna(m3_stat["spent"].median())
m3_stat["city"] = m3_stat["city"].fillna(m3_stat["city"].mode()[0])
m3_stat["education"] = m3_stat["education"].fillna(m3_stat["education"].mode()[0])

print(m3_stat[["income", "age", "spent", "city", "education"]].isna().sum())


income       0
age          0
spent        0
city         0
education    0
dtype: int64


## Missing Values: Method 4 (Forward Fill / Backward Fill)

For ordered or time-based data, missing values can be filled from nearby records:
- forward fill (`ffill`): use previous value
- backward fill (`bfill`): use next value

Use it when row order is meaningful (time, sequence, logs).
Avoid it when rows are independent and unordered.


In [8]:
m4_ffill = df.sort_values("joined_date").copy()
m4_ffill["last_login"] = m4_ffill["last_login"].ffill().bfill()
print("Missing last_login after ffill+bfill:", m4_ffill["last_login"].isna().sum())


Missing last_login after ffill+bfill: 0


## Missing Values: Method 5 (Interpolation)

Interpolation estimates missing numeric values between known points.
It works best for continuous signals that change gradually.

Use it when:
- numeric values follow a smooth trend
- row order is meaningful
- you want an estimated value instead of a constant/statistic

This is common in sensor, time-series, and measurement data.


In [9]:
m5_interp = df.sort_values("customer_id").copy()
m5_interp["spent_interp"] = m5_interp["spent"].interpolate(method="linear")
print("Missing spent before:", m5_interp["spent"].isna().sum())
print("Missing spent_interp after:", m5_interp["spent_interp"].isna().sum())


Missing spent before: 1
Missing spent_interp after: 0


## Missing Values: Method 6 (SimpleImputer from sklearn)

`SimpleImputer` is the sklearn-ready version of common filling strategies.
It fits naturally inside ML pipelines and prevents train/test leakage when used correctly.

Why this is important:
- same transformation can be reused in training and inference
- cleaner integration with `Pipeline` and `ColumnTransformer`
- reproducible preprocessing for deployment


In [10]:
m6_imputer = df.copy()

# Convert None -> np.nan so SimpleImputer handles them
m6_imputer[["city", "education"]] = m6_imputer[["city", "education"]].replace({None: np.nan})

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

m6_imputer[["age", "income", "spent", "purchases"]] = num_imputer.fit_transform(
    m6_imputer[["age", "income", "spent", "purchases"]]
)
m6_imputer[["city", "education"]] = cat_imputer.fit_transform(m6_imputer[["city", "education"]])

print(m6_imputer[["age", "income", "spent", "purchases", "city", "education"]].isna().sum())


age          0
income       0
spent        0
purchases    0
city         0
education    0
dtype: int64


## 4) Choose One Clean Version

Now we select one cleaned dataframe to continue the workflow.
We keep Method 3 for most columns and complete `last_login` with a simple fill.

This step reflects real projects: after trying alternatives, you commit to one consistent preprocessing path.


In [11]:
clean_df = m3_stat.copy()
clean_df["last_login"] = clean_df["last_login"].ffill().bfill()
print(clean_df.isna().sum())


customer_id    0
age            0
income         0
spent          0
purchases      0
city           0
education      0
joined_date    0
last_login     0
churned        0
dtype: int64


## 5) Duplicates Handling

We handle duplicates in two passes:
- remove exact duplicate rows
- resolve duplicate customer IDs by keeping the newest record

Why both checks are needed:
- exact duplicates waste data space and can bias training
- duplicate IDs can create conflicting labels/features

Keeping the latest record is a common business rule for profile data.


In [12]:
# Remove exact duplicates
no_exact = clean_df.drop_duplicates().copy()

# Keep latest row for each customer_id
no_exact = no_exact.sort_values("last_login")
final_unique = no_exact.drop_duplicates(subset=["customer_id"], keep="last")

print("After exact duplicate removal:", no_exact.shape)
print("After customer_id dedup:", final_unique.shape)


After exact duplicate removal: (179, 10)
After customer_id dedup: (178, 10)


## 6) Date Preprocessing

Machine learning models do not directly understand raw datetime objects.
So we convert date columns and extract numeric features such as month and recency.

Features created here:
- `join_month`: seasonal pattern of signup
- `days_since_last_login`: engagement recency

These engineered columns are often more predictive than raw timestamps.


In [13]:
time_df = final_unique.copy()

time_df["joined_date"] = pd.to_datetime(time_df["joined_date"], errors="coerce")
time_df["last_login"] = pd.to_datetime(time_df["last_login"], errors="coerce")

time_df["join_month"] = time_df["joined_date"].dt.month
time_df["days_since_last_login"] = (pd.Timestamp("2025-03-10") - time_df["last_login"]).dt.days

time_df[["joined_date", "last_login", "join_month", "days_since_last_login"]].head()


,joined_date,last_login,join_month,days_since_last_login
0,2024-01-01,2025-02-01,1,37
20,2024-02-10,2025-02-01,2,37
160,2024-11-16,2025-02-01,11,37
140,2024-10-07,2025-02-01,10,37
40,2024-03-21,2025-02-01,3,37


## 7) Categorical Encoding (Multiple Ways)

We demonstrate three encoding strategies:
- one-hot encoding for nominal categories (no natural order)
- ordinal mapping for ordered categories
- label encoding for target labels

Why this matters:
- models require numeric input
- choosing the wrong encoding can hurt performance
- different columns may need different encoding logic


In [14]:
enc_df = time_df.copy()

# One-hot for city
enc_df = pd.get_dummies(enc_df, columns=["city"], drop_first=True)

# Ordinal map for education
edu_map = {"High School": 0, "Bachelor": 1, "Master": 2, "PhD": 3}
enc_df["education_ordinal"] = enc_df["education"].map(edu_map).fillna(1)

# Label encode target
le = LabelEncoder()
enc_df["churned_label"] = le.fit_transform(enc_df["churned"])

print("Target classes:", dict(zip(le.classes_, le.transform(le.classes_))))
enc_df.head()


Target classes: {'No': 0, 'Yes': 1}


,customer_id,age,income,spent,purchases,education,joined_date,last_login,churned,join_month,days_since_last_login,city_Oran,education_ordinal,churned_label
0,1,35.0,152800.0,2820.0,9.0,Master,2024-01-01,2025-02-01,No,1,37,False,2,0
20,21,35.0,118600.0,3000.0,9.0,Master,2024-02-10,2025-02-01,No,2,37,False,2,0
160,161,41.0,87400.0,2370.0,5.0,Bachelor,2024-11-16,2025-02-01,Yes,11,37,True,1,1
140,141,32.0,97000.0,1686.0,4.0,Bachelor,2024-10-07,2025-02-01,Yes,10,37,True,1,1
40,41,36.0,105640.0,3135.0,10.0,Master,2024-03-21,2025-02-01,No,3,37,False,2,0


## 8) Outliers (Simple IQR Capping)

Outliers can distort scaling and model behavior.
We apply IQR capping (winsorization style) to keep extreme values within robust bounds.

Rule used:
- lower bound = Q1 - 1.5 * IQR
- upper bound = Q3 + 1.5 * IQR

This keeps all rows while reducing the influence of extreme points.


In [15]:
out_df = enc_df.copy()

for col in ["age", "income", "spent", "purchases"]:
    q1 = out_df[col].quantile(0.25)
    q3 = out_df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    out_df[col] = out_df[col].clip(low, high)

out_df[["age", "income", "spent", "purchases"]].describe().T


,count,mean,std,min,25%,50%,75%,max
age,178.0,33.129213,6.818688,22.0,28.00,32.0,37.0,50.5
income,178.0,113743.792135,37176.767582,58360.0,85060.00,105640.0,140410.0,223435.0
spent,178.0,2364.337079,562.878698,1182.0,1868.25,2419.5,2820.0,3792.0
purchases,178.0,7.039326,2.003845,4.0,5.00,7.0,9.0,13.0


## 9) Scaling (Two Common Methods)

We build both standardized and min-max scaled versions of key numeric columns.

- `StandardScaler`: centers data around 0 with unit variance
- `MinMaxScaler`: rescales data to [0, 1]

Model note:
- distance/gradient-based models often need scaling
- tree-based models usually need scaling less

Creating both helps you experiment quickly in later modeling notebooks.


In [16]:
scale_df = out_df.copy()
num_cols = ["age", "income", "spent", "purchases", "days_since_last_login"]

std_scaler = StandardScaler()
mm_scaler = MinMaxScaler()

scale_df[[c + "_std" for c in num_cols]] = std_scaler.fit_transform(scale_df[num_cols])
scale_df[[c + "_mm" for c in num_cols]] = mm_scaler.fit_transform(scale_df[num_cols])

scale_df[[c + "_std" for c in num_cols]].head()


,age_std,income_std,spent_std,purchases_std,days_since_last_login_std
0,0.275136,1.053518,0.811806,0.981216,1.603112
20,0.275136,0.130993,1.132493,0.981216,1.603112
160,1.157552,-0.710608,0.010089,-1.020577,1.603112
140,-0.166073,-0.451654,-1.208521,-1.521026,1.603112
40,0.422205,-0.218595,1.373008,1.481665,1.603112


## 10) Final Model-Ready Dataset

Final step:
- select features
- define target
- split into train/test sets

At this point, your data is ready for model training and evaluation.
You can now plug `X_train`, `X_test`, `y_train`, and `y_test` into any classifier or regressor workflow.


In [17]:
model_df = scale_df.copy()

feature_cols = [
    "age_std", "income_std", "spent_std", "purchases_std", "days_since_last_login_std",
    "education_ordinal"
]

# Add city one-hot columns if they exist
feature_cols += [c for c in model_df.columns if c.startswith("city_")]

X = model_df[feature_cols]
y = model_df["churned_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Train size:", X_train.shape, "| Test size:", X_test.shape)
X.head()


X shape: (178, 7)
y shape: (178,)
Train size: (142, 7) | Test size: (36, 7)


,age_std,income_std,spent_std,purchases_std,days_since_last_login_std,education_ordinal,city_Oran
0,0.275136,1.053518,0.811806,0.981216,1.603112,2,False
20,0.275136,0.130993,1.132493,0.981216,1.603112,2,False
160,1.157552,-0.710608,0.010089,-1.020577,1.603112,1,True
140,-0.166073,-0.451654,-1.208521,-1.521026,1.603112,1,True
40,0.422205,-0.218595,1.373008,1.481665,1.603112,2,False
